# Bölüm 17 · Ölçek Uyarlama ve Güvenirlik

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b17-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b17-guvenirlik'
modules=['uygulamalar/b17-guvenirlik/model-belirsizlik']
expected_main=json.loads((root/"bolumler/b17/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 17 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b17/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 17 — görevler

Güncel Revizyon 20 görevleri; önce kendiniz çözün.

## D1–D3: Analizden Önce

1.  Kaynak satırı ile kişi etiketi ne işe yarar? İlk 100 kayıt neden temsilî örneklem sayılmaz?

2.  A1=6 ve C5=1 nasıl ters puanlanır? Eksik A2’ye neden sıfır verilmez?

3.  A, C ve on maddelik ortak analizde n neden farklıdır? Eksik maddeyi silince analiz örnekleminin büyümesi karşılaştırmayı nasıl etkiler?

Yanıtlar Ek F, Bölüm 17’dedir (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## G1–G3: Rehberli Çözüm

1.  A varyanslarından ham alfayı, ortalama r’den standart alfayı hesaplayın. A1 çevrilmeden bulunan değerle farkın anlamı nedir?

2.  A1 için düzeltilmiş ve düzeltilmemiş korelasyonu karşılaştırın. A4 silinirse alfa artışını hesaplayın; bunun istatistiksel anlamlılık veya silme kararı olmadığını açıklayın.

3.  Bootstrap’ta neden hücre değil satır çekilir? \[.464423,.736637\] aralığı Türkçe ölçeğin gerçek güvenirliğini kanıtlar mı?

## B1–B3: C Maddelerinde Bağımsız Çözüm

1.  C4/C5’i anahtara göre çevirin; tam kayıt n’sini, ham/standart alfayı ve ortalama r’yi bulun. Çevirmeden önce negatif alfa ne anlatır?

2.  C maddelerinin madde–kalan ve silinirse alfa tablosunu oluşturun. C5 silmenin kazancı otomatik silmeye yeter mi?

3.  100 sözcüğü aşmayan bir raporda örneklem, puan yönü, eksik kuralı, bootstrap yöntemi, aralık ve kullanım sınırını belirtin.

## H1: Hatalı Raporu Onarma

Her iddiayı bir hesap veya eksik kanıt türüyle düzeltin.

## P1: Uyarlama Kanıt Dosyası

A4 için iki çeviri taslağı, en az üç bilişsel görüşme sorusu ve bir revizyon kayıt şablonu hazırlayın. “Yapıldı/yapılmadı” sütunlarıyla kaynak-haklar, dil, hedef grupta pilot, yapı, güvenirlik, değişmezlik ve kullanım sınırlarını listeleyin. Gerçekte yapılmamış görüşme, örneklem veya uzman onayı uydurmayın. Puanlama: kaynak/puanlama 20; hesap ve eksikler 25; belirsizlik ve madde kararı 20; uyarlama planı 25; yeniden üretim 10.

## T1–T3: Model ve Belirsizlik Denetimi

1.  A ve C için aynı 97 kişinin kullanılmasını ve bootstrap’ta on yanıtın birlikte çekilmesini gerekçelendirin. Fark aralığı sıfırı içerdiğinde ne söylenebilir, ne söylenemez?

2.  Bilinen kongenerik modele .2 artık kovaryansı eklenince alfa ve hedef faktör payı hangi yönde değişir? Hata varyansının 3.1’den 3.5’e çıkışını gösterin.

3.  İkinci bilinen modelin SEM’ini ve bağımsız hata varsayımındaki fark sınırını hesaplayın. Bu sayıyı IPIP katılımcılarına klinik değişim eşiği diye vermek neden yanlıştır?

P1 kanıt dosyasına puanın hedefini, güvenirlik modelini, belirsizlik yöntemini ve grup/birey kararının gerektirdiği ayrı kanıtları ekleyin.

# Alıştırmalar

## Kolay Düzey

1.  Geçerlik ile güvenirlik arasındaki temel fark nedir?

2.  Yüksek Cronbach alfa neden tek başına yeterli görülmez?

3.  Düşük madde–toplam korelasyonu ne düşündürür?

4.  Ters kodlanmış madde düzeltilmezse ne olabilir?

5.  Çok boyutlu bir ölçek için tek alfa değeri vermek neden sorunlu olabilir?

## Orta Düzey

1.  Bir ölçeğin Cronbach alfa katsayısı $0.86$ bulunmuştur. Bulguyu ve sınırlılığını açıklayınız.

2.  Bir maddede düzeltilmiş madde–toplam korelasyonu çok düşük ve “madde silinirse alfa” değeri daha yüksektir. Açıklayınız.

3.  Dört maddeli ölçekte madde varyansları toplamı $5.40$, toplam puan varyansı $15.00$ ise alfa katsayısını hesaplayınız.

4.  30 maddeli bir ölçekte alfa $0.97$ bulunmuştur. Neden otomatik onay yerine ek inceleme gerekir?

5.  Uyarlamada bir madde istatistiksel olarak zayıf ama kuramsal olarak önemlidir. Silmeden önce ne yapılmalıdır?

## İleri Düzey

1.  Üç alt boyutlu ölçekte yalnızca toplam alfa raporlanmıştır. Temel yöntemsel sorun nedir?

2.  Ters kodlama düzeltilince alfa $0.52$’den $0.81$’e çıkmıştır. Nasıl yorumlanır?

3.  Uyarlamada güvenirlik kabul edilebilir, fakat uzmanlar bazı maddelerin hedef kültüre uymadığını belirtmiştir. Güvenirlik neden yetmez?

4.  Araştırmacı alfayı az da olsa artıran her maddeyi silmek istemektedir. Bu strateji neden risklidir?

5.  Güçlü bir güvenirlik raporunda bulunması gereken en az beş unsur yazınız.

Çevrimiçi kaynaklara erişim tarihi: 5 Eylül 2026.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b17/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

In [ ]:
manifest={"chapter":17,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b17-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b17-guvenirlik',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)